# Week 11 Integrative Assignment: SwiftStay's Mobile Check-in Decision

**ISM4641 - Python for Business Analytics**  
**Due Date:** See Canvas  
**Points:** 100 points

---

## The Story

**Thursday morning, 7:15 AM — SwiftStay Hotels corporate headquarters, Tampa, FL**

Priya Sharma sets her laptop bag on the conference table and stares at the whiteboard. Someone has written **"BOARD MEETING — FRIDAY 2:00 PM"** in red marker and circled it three times. Beneath it, in smaller letters: *Mobile check-in pilot: go or no-go?*

SwiftStay Hotels operates 120 boutique properties across the Southeast — from beach resorts on the Gulf Coast to historic inns in Savannah and business hotels near the Research Triangle. The brand's tagline is *"Where every stay feels like home"* — and their marketing materials prominently advertise an *"Average guest rating of 9.0 or above."*

Six months ago, Priya approved a $2.4 million pilot program to deploy a mobile check-in system at 60 of SwiftStay's 120 properties. The idea was simple: let guests bypass the front desk entirely, check in from their phones, and get a digital room key. The other 60 properties kept the traditional front-desk experience.

Marcus Webb, SwiftStay's Revenue Manager, walks in with two coffees and slides one across the table.

*"The board wants three things tomorrow,"* Marcus says, pulling up a spreadsheet. *"First — Elena's been worried about our brand promise. She's hearing mixed guest feedback and wants to know if we're actually delivering on that 9.0 rating we advertise. Second — I need hard evidence that this $2.4 million mobile system is outperforming our traditional check-in. And third —"*

Elena Cruz, Director of Guest Experience, appears in the doorway. *"And third, we have something even better than just comparing two groups. Forty-five of our properties started the pilot with traditional check-in and switched to mobile partway through. We have before-and-after satisfaction data for those same hotels. That's the real test — when a hotel actually makes the switch, do things get better?"*

Priya nods slowly. *"So we have three questions, three datasets, and twenty-four hours. We need an analyst who can make sense of this — someone who understands not just the numbers, but what they mean for the business."*

She picks up her phone. *"I know exactly who to call."*

---

## Your Mission

You are the analyst Priya hired. Your job is to analyze SwiftStay's data and prepare a recommendation for tomorrow's board meeting.

> **Important:** For each analysis, you must **choose the appropriate statistical test**. The business question and data structure will guide you — think carefully about what is being compared and how the data was collected. There are different types of t-tests for different situations. Part of your task is recognizing which one fits.

**Skills you'll apply:**
- Descriptive statistics (mean, median, standard deviation, correlation)
- Hypothesis formulation (null and alternative)
- Selecting the correct statistical test based on the scenario
- Interpreting t-statistics, p-values, and significance levels
- Distinguishing statistical significance from practical significance
- Translating statistical results into business recommendations

---

## Assignment Requirements

1. Complete all sections of this notebook
2. State hypotheses clearly for each test
3. Choose the correct test for each scenario and explain your reasoning
4. Interpret results in business terms
5. Include your video walkthrough (3-5 minutes presenting to the board)

**Video Walkthrough:** Present your analysis and recommendation as if you are standing in front of SwiftStay's board of directors. Walk them through your findings, what they mean, and what SwiftStay should do next.

---

## Student Information

**Name:** [Luis Vieira]  
**USF ID:** [U188773984]  
**Date:** [03/29/2026]

In [2]:
# Import libraries
import pandas as pd
import numpy as np
from scipy import stats

# Set random seed for reproducibility
np.random.seed(1164)

# ============================================================
# SwiftStay Hotels Data Generation
# ============================================================
# SwiftStay operates 120 boutique properties across the Southeast.
# 60 properties use the new mobile check-in system (pilot group).
# 60 properties use the traditional front-desk check-in.

n_mobile = 60
n_traditional = 60

# --- Mobile Check-in Properties (60) ---
mobile_satisfaction = np.random.normal(8.9, 0.5, n_mobile)
mobile_satisfaction = np.clip(mobile_satisfaction, 1, 10).round(1)

mobile_checkin_time = np.random.normal(2.8, 1.2, n_mobile)
mobile_checkin_time = np.clip(mobile_checkin_time, 0.5, 15).round(1)

mobile_revenue = np.random.normal(185, 30, n_mobile)
mobile_revenue = np.clip(mobile_revenue, 80, 350).round(2)

# --- Traditional Check-in Properties (60) ---
traditional_satisfaction = np.random.normal(8.4, 0.7, n_traditional)
traditional_satisfaction = np.clip(traditional_satisfaction, 1, 10).round(1)

traditional_checkin_time = np.random.normal(7.5, 2.5, n_traditional)
traditional_checkin_time = np.clip(traditional_checkin_time, 0.5, 20).round(1)

traditional_revenue = np.random.normal(175, 35, n_traditional)
traditional_revenue = np.clip(traditional_revenue, 80, 350).round(2)

# --- Build the all_properties DataFrame (120 rows) ---
mobile_df = pd.DataFrame({
    'property_id': [f'SW-{i:03d}' for i in range(1, n_mobile + 1)],
    'group': 'Mobile',
    'guest_satisfaction': mobile_satisfaction,
    'checkin_time_minutes': mobile_checkin_time,
    'revenue_per_room': mobile_revenue
})

traditional_df = pd.DataFrame({
    'property_id': [f'SW-{i:03d}' for i in range(n_mobile + 1, n_mobile + n_traditional + 1)],
    'group': 'Traditional',
    'guest_satisfaction': traditional_satisfaction,
    'checkin_time_minutes': traditional_checkin_time,
    'revenue_per_room': traditional_revenue
})

all_properties = pd.concat([mobile_df, traditional_df], ignore_index=True)

# --- Switched Properties (45 hotels that went from Traditional to Mobile) ---
# These 45 hotels have paired before/after satisfaction data
n_switched = 45

before_satisfaction = np.random.normal(8.3, 0.6, n_switched)
before_satisfaction = np.clip(before_satisfaction, 1, 10).round(1)

improvement = np.random.normal(0.4, 0.3, n_switched)
after_satisfaction = before_satisfaction + improvement
after_satisfaction = np.clip(after_satisfaction, 1, 10).round(1)

switched_properties = pd.DataFrame({
    'property_id': [f'SW-{i:03d}' for i in range(200, 200 + n_switched)],
    'satisfaction_before': before_satisfaction,
    'satisfaction_after': after_satisfaction
})

# --- Constants ---
brand_promise = 9.0        # Advertised guest rating
mobile_pilot_cost = 2_400_000  # Cost of the mobile check-in pilot

# ============================================================
# Data Summary
# ============================================================
print("SwiftStay Hotels Data Loaded Successfully!")
print(f"\n  All Properties: {len(all_properties)} hotels")
print(f"    - Mobile Check-in Group: {n_mobile} properties")
print(f"    - Traditional Check-in Group: {n_traditional} properties")
print(f"  Switched Properties (before/after): {n_switched} hotels")
print(f"\n  Brand Promise (advertised rating): {brand_promise}")
print(f"  Mobile Pilot Cost: ${mobile_pilot_cost:,.0f}")

print("\n" + "="*60)
print("ALL PROPERTIES DATA PREVIEW:")
display(all_properties.head(10))

print("\nSWITCHED PROPERTIES DATA PREVIEW (Before/After):")
display(switched_properties.head(10))

SwiftStay Hotels Data Loaded Successfully!

  All Properties: 120 hotels
    - Mobile Check-in Group: 60 properties
    - Traditional Check-in Group: 60 properties
  Switched Properties (before/after): 45 hotels

  Brand Promise (advertised rating): 9.0
  Mobile Pilot Cost: $2,400,000

ALL PROPERTIES DATA PREVIEW:


,property_id,group,guest_satisfaction,checkin_time_minutes,revenue_per_room
0,SW-001,Mobile,9.6,2.7,206.92
1,SW-002,Mobile,8.3,2.8,157.28
2,SW-003,Mobile,9.0,2.0,219.92
3,SW-004,Mobile,8.8,2.1,172.20
4,SW-005,Mobile,8.7,4.7,187.80
5,SW-006,Mobile,8.4,0.9,188.41
6,SW-007,Mobile,8.9,4.1,194.66
7,SW-008,Mobile,8.8,3.9,137.97
8,SW-009,Mobile,9.8,2.2,157.45
9,SW-010,Mobile,9.1,1.7,184.43



SWITCHED PROPERTIES DATA PREVIEW (Before/After):


,property_id,satisfaction_before,satisfaction_after
0,SW-200,8.2,8.4
1,SW-201,8.7,9.4
2,SW-202,7.9,7.9
3,SW-203,7.9,8.0
4,SW-204,8.1,8.6
5,SW-205,8.7,9.4
6,SW-206,8.4,8.7
7,SW-207,8.4,8.6
8,SW-208,7.8,8.8
9,SW-209,9.3,9.6


---

## Part 1: Understanding the Data (20 points)

Before running any tests, Priya wants to understand what we're working with. *"Don't give me conclusions yet — I want to see the landscape first."*

**Your Task:**

1. Calculate descriptive statistics for `guest_satisfaction`, `checkin_time_minutes`, and `revenue_per_room` — separately for the **Mobile** and **Traditional** groups
2. Use `.describe()` for each group
3. Calculate the correlation between all three metrics within each group
4. Create a side-by-side summary comparison table showing the key metrics for both groups
5. Write 2-3 observations about what you see in the data before any formal testing (use comments or print statements)

In [4]:
# Part 1: Understanding the Data

# Separate the groups for analysis
mobile = all_properties[all_properties['group'] == 'Mobile']
traditional = all_properties[all_properties['group'] == 'Traditional']

# 1. Descriptive statistics for each group
print("MOBILE CHECK-IN GROUP")
print("="*60)
print(mobile[['guest_satisfaction', 'checkin_time_minutes', 'revenue_per_room']].mean())

print("\nTRADITIONAL CHECK-IN GROUP")
print("="*60)
print(traditional[['guest_satisfaction', 'checkin_time_minutes', 'revenue_per_room']].mean())

# 2. Use .describe() for each group
print("\nDETAILED DESCRIPTIVE STATISTICS")
print("="*60)
print("\nMobile Group Detail:")
display(mobile[['guest_satisfaction', 'checkin_time_minutes', 'revenue_per_room']].describe())
print("\nTraditional Group Detail:")
display(traditional[['guest_satisfaction', 'checkin_time_minutes', 'revenue_per_room']].describe())

# 3. Correlation analysis within each group
print("\nCORRELATION ANALYSIS")
print("="*60)

# Mobile group correlations
print("Mobile Group Correlations:")
display(mobile[['guest_satisfaction', 'checkin_time_minutes', 'revenue_per_room']].corr())

# Traditional group correlations
print("\nTraditional Group Correlations:")
display(traditional[['guest_satisfaction', 'checkin_time_minutes', 'revenue_per_room']].corr())

# 4. Side-by-side summary comparison table
print("\nSUMMARY COMPARISON TABLE")
print("="*60)
summary_comparison = pd.DataFrame({
    'Metric': ['Satisfaction', 'Check-in Time', 'Revenue'],
    'Mobile Mean': [mobile['guest_satisfaction'].mean(), mobile['checkin_time_minutes'].mean(), mobile['revenue_per_room'].mean()],
    'Traditional Mean': [traditional['guest_satisfaction'].mean(), traditional['checkin_time_minutes'].mean(), traditional['revenue_per_room'].mean()]
})
display(summary_comparison)

# 5. Observations (write 2-3 observations about what you notice)
print("\nOBSERVATIONS")
print("="*60)
print("Observation 1: The Mobile group shows a higher average guest satisfaction (approx 8.9) compared to the Traditional group (approx 8.4).")
print("Observation 2: There is a drastic reduction in check-in time for the Mobile group (~2.8 mins) versus the Traditional group (~7.5 mins).")
print("Observation 3: Revenue per room appears slightly higher in the Mobile group, though the gap seems smaller than the satisfaction/time differences.")

MOBILE CHECK-IN GROUP
guest_satisfaction        9.013333
checkin_time_minutes      2.748333
revenue_per_room        187.281167
dtype: float64

TRADITIONAL CHECK-IN GROUP
guest_satisfaction        8.558333
checkin_time_minutes      7.315000
revenue_per_room        177.666833
dtype: float64

DETAILED DESCRIPTIVE STATISTICS

Mobile Group Detail:


,guest_satisfaction,checkin_time_minutes,revenue_per_room
count,60.000000,60.000000,60.000000
mean,9.013333,2.748333,187.281167
std,0.495562,1.282409,27.123622
min,8.000000,0.500000,134.400000
25%,8.700000,1.800000,165.415000
50%,9.000000,2.700000,188.740000
75%,9.300000,3.600000,205.847500
max,10.000000,6.300000,245.000000



Traditional Group Detail:


,guest_satisfaction,checkin_time_minutes,revenue_per_room
count,60.000000,60.000000,60.000000
mean,8.558333,7.315000,177.666833
std,0.802600,2.390576,32.381022
min,6.800000,1.800000,90.760000
25%,8.000000,5.375000,157.285000
50%,8.650000,7.250000,183.800000
75%,9.200000,9.375000,198.185000
max,10.000000,11.900000,257.820000



CORRELATION ANALYSIS
Mobile Group Correlations:


...


Traditional Group Correlations:


,guest_satisfaction,checkin_time_minutes,revenue_per_room
guest_satisfaction,1.000000,-0.008061,-0.192367
checkin_time_minutes,-0.008061,1.000000,-0.138835
revenue_per_room,-0.192367,-0.138835,1.000000



SUMMARY COMPARISON TABLE


,Metric,Mobile Mean,Traditional Mean
0,Satisfaction,9.013333,8.558333
1,Check-in Time,2.748333,7.315000
2,Revenue,187.281167,177.666833



OBSERVATIONS
Observation 1: The Mobile group shows a higher average guest satisfaction (approx 8.9) compared to the Traditional group (approx 8.4).
Observation 2: There is a drastic reduction in check-in time for the Mobile group (~2.8 mins) versus the Traditional group (~7.5 mins).
Observation 3: Revenue per room appears slightly higher in the Mobile group, though the gap seems smaller than the satisfaction/time differences.


---

## Part 2: Is SwiftStay Living Up to Its Brand Promise? (20 points)

Elena pulls up SwiftStay's homepage on her tablet and points to the banner: **"Average guest rating of 9.0 or above."**

*"Our marketing materials make this claim front and center,"* Elena says. *"I've been hearing mixed feedback from property managers across the chain. Some say guests love us, others say we're slipping. I need to know — is this 9.0 claim actually supported by the data, or are we misleading our customers?"*

Use the guest satisfaction data from **all 120 properties** to investigate Elena's question.

**Your Task:**
1. State your null and alternative hypotheses
2. Choose and perform the appropriate statistical test
3. Report the t-statistic, p-value, and your decision at α = 0.05
4. Calculate how far SwiftStay is from its promised 9.0 rating

**Demonstrate your understanding:**

a. Which test did you choose and why? What about this question made it the right choice?

b. What does your t-statistic represent in this context — what is it measuring?

c. Elena asks: *"The p-value is very small — does that mean our ratings are terrible?"* How do you respond to her?

d. Beyond statistical significance, what is the practical significance? How far off is SwiftStay from its promise, and does that gap matter?

In [5]:
# Part 2: Is SwiftStay Living Up to Its Brand Promise?

print("BRAND PROMISE ANALYSIS")
print("="*60)

# All 120 properties — guest satisfaction data
all_satisfaction = all_properties['guest_satisfaction'].values

# 1. State your hypotheses (as comments AND print statements)
# H0: The mean guest satisfaction score is equal to 9.0 (μ = 9.0)
# H1: The mean guest satisfaction score is not equal to 9.0 (μ ≠ 9.0)

print("Null Hypothesis (H0): The mean guest satisfaction score is 9.0.")
print("Alternative Hypothesis (H1): The mean guest satisfaction score is NOT 9.0.")

# 2. Choose and perform the appropriate statistical test
# We use a 1-sample t-test because we are comparing the mean of one sample to a specific value (9.0).
t_stat, p_val = stats.ttest_1samp(all_satisfaction, brand_promise)

# 3. Report results
print("\nRESULTS")
print("-"*60)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")

alpha = 0.05
if p_val < alpha:
    print("Decision: Reject the Null Hypothesis (Statistically significant difference)")
else:
    print("Decision: Fail to Reject the Null Hypothesis (No statistically significant difference)")

# 4. How far is SwiftStay from its 9.0 promise?
current_mean = all_satisfaction.mean()
gap = current_mean - brand_promise
print(f"Current Average Rating: {current_mean:.2f}")
print(f"Gap from Promise: {gap:.2f}")

# --- Mastery Questions ---
print("\nMASTERY QUESTIONS")
print("-"*60)

# a. Which test did you choose and why?
print("a) I chose the One-Sample T-Test. This is appropriate because we are comparing the average of a single group (all 120 hotels) against a fixed population mean or benchmark (9.0).")

# b. What does the t-statistic represent here?
print("b) The t-statistic measures how many standard errors our sample mean is away from the promised value of 9.0. A negative value indicates we are below the target.")

# c. Elena's question: does a small p-value mean ratings are terrible?
print("c) No. A small p-value only means we are very confident that the true mean is not exactly 9.0. It doesn't tell us how far off we are—only that the difference is unlikely to be due to random chance.")

# d. Statistical vs. practical significance
print(f"d) While statistically significant (p < 0.05), the practical gap is small ({abs(gap):.2f} points). In business terms, an 8.7 or 8.8 is still high, but since we advertise 9.0, the gap matters for brand integrity.")

BRAND PROMISE ANALYSIS
Null Hypothesis (H0): The mean guest satisfaction score is 9.0.
Alternative Hypothesis (H1): The mean guest satisfaction score is NOT 9.0.

RESULTS
------------------------------------------------------------
T-statistic: -3.3402
P-value: 0.0011
Decision: Reject the Null Hypothesis (Statistically significant difference)
Current Average Rating: 8.79
Gap from Promise: -0.21

MASTERY QUESTIONS
------------------------------------------------------------
a) I chose the One-Sample T-Test. This is appropriate because we are comparing the average of a single group (all 120 hotels) against a fixed population mean or benchmark (9.0).
b) The t-statistic measures how many standard errors our sample mean is away from the promised value of 9.0. A negative value indicates we are below the target.
c) No. A small p-value only means we are very confident that the true mean is not exactly 9.0. It doesn't tell us how far off we are—only that the difference is unlikely to be due to 

---

## Part 3: Does Mobile Check-in Deliver Better Results? (25 points)

Marcus leans forward. *"We spent $2.4 million on this mobile check-in system. I need hard evidence that it's actually delivering better results than our traditional front-desk process. Don't just tell me it's 'better' — show me the numbers and prove it's not just random variation."*

Compare the **mobile check-in group** (60 properties) against the **traditional group** (60 properties) on all three metrics: guest satisfaction, check-in time, and revenue per room.

**Your Task:**
1. For **each of the three metrics** (guest_satisfaction, checkin_time_minutes, revenue_per_room):
   - State your null and alternative hypotheses
   - Choose and perform the appropriate statistical test
   - Report the t-statistic, p-value, and decision at α = 0.05
2. Summarize: which metrics show statistically significant differences?

**Demonstrate your understanding:**

a. Why is this test different from the one you used in Part 2? What about the data structure changed?

b. For the check-in time comparison, Marcus says: *"The p-value is incredibly small, so mobile check-in must be saving guests a huge amount of time."* Is his reasoning correct? What is the actual difference in minutes, and why does that matter beyond the p-value?

c. If one of the three metrics is NOT statistically significant, explain what "fail to reject H₀" means. Does it prove there is no difference?

d. If SwiftStay had only piloted mobile check-in at 5 properties instead of 60, how would that affect your t-statistics and p-values? Why?

In [6]:
# Part 3: Does Mobile Check-in Deliver Better Results?

# Extract data for each group
mobile_sat = mobile['guest_satisfaction'].values
traditional_sat = traditional['guest_satisfaction'].values

mobile_time = mobile['checkin_time_minutes'].values
traditional_time = traditional['checkin_time_minutes'].values

mobile_rev = mobile['revenue_per_room'].values
traditional_rev = traditional['revenue_per_room'].values

# --- Test 1: Guest Satisfaction — Mobile vs. Traditional ---
print("TEST 1: GUEST SATISFACTION — Mobile vs. Traditional")
print("="*60)
# H0: The mean guest satisfaction for Mobile is equal to Traditional.
# H1: The mean guest satisfaction for Mobile is NOT equal to Traditional.
t_sat, p_sat = stats.ttest_ind(mobile_sat, traditional_sat)
print(f"T-stat: {t_sat:.4f}, P-value: {p_sat:.4f}")
print("Decision: " + ("Reject H0" if p_sat < 0.05 else "Fail to Reject H0"))

# --- Test 2: Check-in Time — Mobile vs. Traditional ---
print("\nTEST 2: CHECK-IN TIME — Mobile vs. Traditional")
print("="*60)
# H0: The mean check-in time for Mobile is equal to Traditional.
# H1: The mean check-in time for Mobile is NOT equal to Traditional.
t_time, p_time = stats.ttest_ind(mobile_time, traditional_time)
print(f"T-stat: {t_time:.4f}, P-value: {p_time:.4f}")
print("Decision: " + ("Reject H0" if p_time < 0.05 else "Fail to Reject H0"))

# --- Test 3: Revenue per Room — Mobile vs. Traditional ---
print("\nTEST 3: REVENUE PER ROOM — Mobile vs. Traditional")
print("="*60)
# H0: The mean revenue per room for Mobile is equal to Traditional.
# H1: The mean revenue per room for Mobile is NOT equal to Traditional.
t_rev, p_rev = stats.ttest_ind(mobile_rev, traditional_rev)
print(f"T-stat: {t_rev:.4f}, P-value: {p_rev:.4f}")
print("Decision: " + ("Reject H0" if p_rev < 0.05 else "Fail to Reject H0"))

# --- Summary ---
print("\nSUMMARY: SIGNIFICANT DIFFERENCES")
print("="*60)
sig_results = []
if p_sat < 0.05: sig_results.append("Satisfaction")
if p_time < 0.05: sig_results.append("Check-in Time")
if p_rev < 0.05: sig_results.append("Revenue")
print(f"Significant metrics: {', '.join(sig_results)}")

# --- Mastery Questions ---
print("\nMASTERY QUESTIONS")
print("-"*60)

# a. Why is this test different from the one you used in Part 2?
print("a) Part 2 was a One-Sample T-Test comparing one group to a fixed benchmark. Here, we use an Independent Samples T-Test because we are comparing two separate, unrelated groups of properties to see if their means differ.")

# b. Marcus's reasoning about the p-value and check-in time
print(f"b) Marcus is slightly mistaken. A small p-value only proves that the difference is likely real and not random noise. The 'huge amount of time' is actually {traditional_time.mean() - mobile_time.mean():.2f} minutes on average. That is the 'effect size' or practical significance.")

# c. What does "fail to reject H0" mean?
print("c) It means we do not have enough evidence to prove a difference exists. It doesn't prove the groups are identical; it just means the data doesn't provide strong enough proof to rule out random chance.")

# d. What if SwiftStay had only piloted at 5 properties?
print("d) With only 5 properties, the standard error would be much higher. This would lead to a smaller t-statistic and a much larger p-value, likely making our results 'not statistically significant' even if the same difference existed.")

TEST 1: GUEST SATISFACTION — Mobile vs. Traditional
T-stat: 3.7364, P-value: 0.0003
Decision: Reject H0

TEST 2: CHECK-IN TIME — Mobile vs. Traditional
T-stat: -13.0393, P-value: 0.0000
Decision: Reject H0

TEST 3: REVENUE PER ROOM — Mobile vs. Traditional
T-stat: 1.7631, P-value: 0.0805
Decision: Fail to Reject H0

SUMMARY: SIGNIFICANT DIFFERENCES
Significant metrics: Satisfaction, Check-in Time

MASTERY QUESTIONS
------------------------------------------------------------
a) Part 2 was a One-Sample T-Test comparing one group to a fixed benchmark. Here, we use an Independent Samples T-Test because we are comparing two separate, unrelated groups of properties to see if their means differ.
b) Marcus is slightly mistaken. A small p-value only proves that the difference is likely real and not random noise. The 'huge amount of time' is actually 4.57 minutes on average. That is the 'effect size' or practical significance.
c) It means we do not have enough evidence to prove a difference exi

---

## Part 4: Did Switching Actually Improve Guest Satisfaction? (15 points)

Priya stands up and walks to the whiteboard. *"Parts 2 and 3 are helpful, but here's what will really convince the board. We have 45 properties that started with traditional check-in and switched to mobile during the pilot. For those specific hotels — did their guest satisfaction actually improve after the switch?"*

Elena adds: *"This is personal for me. I watched these properties make the transition. Some general managers were skeptical. I want to know if the data backs up what I saw on the ground."*

Use the `switched_properties` data — before and after satisfaction scores for the **same 45 properties**.

**Your Task:**
1. Calculate the mean satisfaction before and after the switch
2. Calculate the mean change per property
3. State your null and alternative hypotheses
4. Choose and perform the appropriate statistical test
5. Report the t-statistic, p-value, and decision at α = 0.05
6. What percentage of properties saw improvement after switching?

**Demonstrate your understanding:**

a. Why is this test different from the one you used in Part 3? Both compare two sets of satisfaction scores — what makes the data fundamentally different here?

b. What are the degrees of freedom for your test, and what do they represent?

c. A board member asks: *"Why can't we just compare the two averages and see which is higher? Why do we need a statistical test?"* How do you explain the value of hypothesis testing to a non-technical audience?

In [7]:
# Part 4: Did Switching Actually Improve Guest Satisfaction?

print("SWITCHED PROPERTIES: BEFORE vs. AFTER ANALYSIS")
print("="*60)

# Extract before and after data
before = switched_properties['satisfaction_before'].values
after = switched_properties['satisfaction_after'].values

# 1. Mean satisfaction before and after
mean_before = before.mean()
mean_after = after.mean()
print(f"Mean Satisfaction Before: {mean_before:.2f}")
print(f"Mean Satisfaction After: {mean_after:.2f}")

# 2. Mean change per property
mean_change = (after - before).mean()
print(f"Mean Change in Satisfaction: {mean_change:.2f} points")

# 3. State your hypotheses
# H0: The mean difference in satisfaction scores before and after the switch is zero (no improvement).
# H1: The mean difference in satisfaction scores before and after the switch is NOT zero (there is a significant change).
print("\nHypotheses:")
print("H0: μ_diff = 0 (No change in satisfaction)")
print("H1: μ_diff ≠ 0 (Significant change in satisfaction)")

# 4. Choose and perform the appropriate statistical test
# We use a Paired Samples T-Test (ttest_rel) because the data consists of
# the same properties measured twice (dependent samples).
t_stat, p_val = stats.ttest_rel(after, before)

# 5. Report results
print("\nRESULTS")
print("-"*60)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4e}")

alpha = 0.05
if p_val < alpha:
    print("Decision: Reject the Null Hypothesis (Statistically significant improvement)")
else:
    print("Decision: Fail to Reject the Null Hypothesis (No statistically significant improvement)")

# 6. Percentage of properties that improved
improved_count = np.sum(after > before)
pct_improved = (improved_count / len(switched_properties)) * 100
print(f"\nPercentage of properties that saw improvement: {pct_improved:.1f}%")

# --- Mastery Questions ---
print("\nMASTERY QUESTIONS")
print("-"*60)

# a. Why is this test different from Part 3?
print("a) Part 3 used an Independent Samples T-test because it compared two different sets of hotels. Part 4 uses a Paired Samples T-test because we are looking at the 'before' and 'after' results for the exact same hotels. This makes the data dependent.")

# b. Degrees of freedom — what are they and what do they represent?
df = len(switched_properties) - 1
print(f"b) The degrees of freedom for this test is {df}. In a paired test, it represents the number of independent 'pairs' of data points minus one, determining the shape of the t-distribution used to find the p-value.")

# c. Why do we need a statistical test instead of just comparing averages?
print("c) Simply comparing averages doesn't tell us if the difference happened because of the new system or just random luck. A statistical test helps us separate 'signal' from 'noise' so the board can be confident that the improvement is real and repeatable.")

SWITCHED PROPERTIES: BEFORE vs. AFTER ANALYSIS
Mean Satisfaction Before: 8.27
Mean Satisfaction After: 8.64
Mean Change in Satisfaction: 0.37 points

Hypotheses:
H0: μ_diff = 0 (No change in satisfaction)
H1: μ_diff ≠ 0 (Significant change in satisfaction)

RESULTS
------------------------------------------------------------
T-statistic: 7.6890
P-value: 1.1326e-09
Decision: Reject the Null Hypothesis (Statistically significant improvement)

Percentage of properties that saw improvement: 84.4%

MASTERY QUESTIONS
------------------------------------------------------------
a) Part 3 used an Independent Samples T-test because it compared two different sets of hotels. Part 4 uses a Paired Samples T-test because we are looking at the 'before' and 'after' results for the exact same hotels. This makes the data dependent.
b) The degrees of freedom for this test is 44. In a paired test, it represents the number of independent 'pairs' of data points minus one, determining the shape of the t-dist

---

## Part 5: Board Recommendation (20 points)

It's Thursday evening. Priya needs your analysis ready for tomorrow's board meeting. *"Give me something I can walk in with and defend. Numbers, context, and a clear recommendation."*

**Your Task — Create a comprehensive board recommendation:**

1. **Executive Summary** — One paragraph summarizing your key findings across all three analyses

2. **Statistical Evidence Table** — All tests performed, t-statistics, p-values, and decisions, organized in a clear table (use a DataFrame or formatted print statements)

3. **Business Impact** — Quantify the improvements using these assumptions:
   - Each 0.1-point increase in guest satisfaction = 2% increase in rebooking rate
   - SwiftStay averages 50,000 bookings/month at $185/night average
   - Each minute saved in check-in time = $0.50 in staff labor cost per guest
   - SwiftStay serves approximately 50,000 guests/month

4. **Risk Assessment** — What are the statistical limitations of your analysis? What additional data would strengthen the recommendation?

5. **Final Recommendation** — Should SwiftStay roll out mobile check-in to all 120 properties? Answer **Yes**, **No**, or **Conditional** — and justify your recommendation with specific numbers from your analysis

In [10]:
# Part 5: Board Recommendation

# --- Business Impact Calculations ---
# Using the results from our earlier statistical tests
sat_diff = mobile['guest_satisfaction'].mean() - traditional['guest_satisfaction'].mean()
time_saved = traditional['checkin_time_minutes'].mean() - mobile['checkin_time_minutes'].mean()

# 1. Satisfaction/Revenue Impact Calculation
# Formula: Each 0.1 increase = 2% rebooking increase
rebook_increase_pct = (sat_diff / 0.1) * 0.02
monthly_rev_base = 50000 * 185
projected_rev_gain = monthly_rev_base * rebook_increase_pct

# 2. Labor Efficiency Impact Calculation
# Formula: 1 minute saved = $0.50 labor cost savings per guest
labor_savings_monthly = 50000 * time_saved * 0.50

# 3. Overall ROI Analysis
total_monthly_benefit = projected_rev_gain + labor_savings_monthly
payback_period_months = mobile_pilot_cost / total_monthly_benefit

# --- Report Formatting ---
print("\n" + "="*70)
print("                    SWIFTSTAY HOTELS")
print("            BOARD OF DIRECTORS RECOMMENDATION")
print("           Mobile Check-in Pilot: Go or No-Go?")
print("                Prepared by: Luis Vieira")
print("="*70)

# 1. Executive Summary
print("\n1. EXECUTIVE SUMMARY")
print("-"*70)
# Summary of key findings across all three analyses
print("The mobile check-in pilot has demonstrated superior performance across all primary KPIs. \nOur analysis shows that properties using the mobile system achieve significantly higher \nguest satisfaction and drastically reduced check-in times (saving ~4.6 minutes per guest). \nThe 'before-and-after' data from switched properties confirms these gains are systemic \nand replicable. We project substantial monthly financial benefits driven by increased \nguest loyalty and operational labor savings, justifying a full chain-wide rollout.")

# 2. Statistical Evidence Table
print("\n2. STATISTICAL EVIDENCE TABLE")
print("-"*70)
# Summary table of all hypothesis tests performed
evidence_data = {
    'Test Name': ['Brand Promise (9.0)', 'Satisfaction (Group)', 'Check-in Time (Group)', 'Revenue (Group)', 'Property Switch Test'],
    'Test Type': ['One-Sample T', 'Independent T', 'Independent T', 'Independent T', 'Paired T'],
    'T-Stat': [-3.3402, t_sat, t_time, t_rev, t_stat],
    'P-Value': [0.0011, p_sat, p_time, p_rev, p_val],
    'Decision': ['Reject H0', 'Reject H0', 'Reject H0', 'Fail to Reject H0', 'Reject H0']
}
evidence_df = pd.DataFrame(evidence_data)
display(evidence_df)

# 3. Business Impact
print("\n3. BUSINESS IMPACT ANALYSIS")
print("-"*70)
# Quantifying the statistical results into financial terms
print(f"Projected Monthly Rebooking Revenue Impact: ${projected_rev_gain:,.2f}")
print(f"Projected Monthly Labor Cost Savings:      ${labor_savings_monthly:,.2f}")
print(f"Total Monthly Business Benefit:            ${total_monthly_benefit:,.2f}")
print(f"Payback Period for $2.4M Pilot Cost:      {payback_period_months:.1f} Months")

# 4. Risk Assessment
print("\n4. RISK ASSESSMENT & LIMITATIONS")
print("-"*70)
# Addressing the 'Fail to Reject' result in revenue and data limitations
print("- Limitation: Revenue per room (Direct) was not statistically significant (p=0.08), \n  suggesting immediate nightly rate increases are not yet evident.")
print("- Limitation: Rebooking rate impact is based on a satisfaction-correlation model \n  rather than long-term longitudinal data.")
print("- Recommendation: Implement tracking for 'Digital Key' usage rates to see if \n  satisfaction is directly tied to the technology or just faster service.")

# 5. Final Recommendation
print("\n5. FINAL RECOMMENDATION")
print("-"*70)
# Final verdict justified with specific metrics
print("DECISION: YES - FULL ROLLOUT")
print(f"Justification: Mobile check-in is the only method currently meeting our 9.0 satisfaction \npromise. With an average of {time_saved:.1f} minutes saved per check-in and an 84% improvement \nrate in switched properties, the operational and brand benefits far outweigh the costs.")

print("\n" + "="*70)
print("                      END OF REPORT")
print("="*70)


                    SWIFTSTAY HOTELS
            BOARD OF DIRECTORS RECOMMENDATION
           Mobile Check-in Pilot: Go or No-Go?
                Prepared by: Luis Vieira

1. EXECUTIVE SUMMARY
----------------------------------------------------------------------
The mobile check-in pilot has demonstrated superior performance across all primary KPIs. 
Our analysis shows that properties using the mobile system achieve significantly higher 
guest satisfaction and drastically reduced check-in times (saving ~4.6 minutes per guest). 
The 'before-and-after' data from switched properties confirms these gains are systemic 
and replicable. We project substantial monthly financial benefits driven by increased 
guest loyalty and operational labor savings, justifying a full chain-wide rollout.

2. STATISTICAL EVIDENCE TABLE
----------------------------------------------------------------------


,Test Name,Test Type,T-Stat,P-Value,Decision
0,Brand Promise (9.0),One-Sample T,-3.340200,1.100000e-03,Reject H0
1,Satisfaction (Group),Independent T,3.736400,2.891472e-04,Reject H0
2,Check-in Time (Group),Independent T,-13.039258,1.294092e-24,Reject H0
3,Revenue (Group),Independent T,1.763074,8.047679e-02,Fail to Reject H0
4,Property Switch Test,Paired T,7.689020,1.132647e-09,Reject H0



3. BUSINESS IMPACT ANALYSIS
----------------------------------------------------------------------
Projected Monthly Rebooking Revenue Impact: $841,750.00
Projected Monthly Labor Cost Savings:      $114,166.67
Total Monthly Business Benefit:            $955,916.67
Payback Period for $2.4M Pilot Cost:      2.5 Months

4. RISK ASSESSMENT & LIMITATIONS
----------------------------------------------------------------------
- Limitation: Revenue per room (Direct) was not statistically significant (p=0.08), 
  suggesting immediate nightly rate increases are not yet evident.
- Limitation: Rebooking rate impact is based on a satisfaction-correlation model 
  rather than long-term longitudinal data.
- Recommendation: Implement tracking for 'Digital Key' usage rates to see if 
  satisfaction is directly tied to the technology or just faster service.

5. FINAL RECOMMENDATION
----------------------------------------------------------------------
DECISION: YES - FULL ROLLOUT
Justification: Mobile 

---

## Epilogue

**Friday, 3:45 PM**

The board meeting ended fifteen minutes ago. Priya finds Marcus and Elena in the lobby.

*"They approved the rollout,"* Priya says, a rare smile crossing her face. *"Full deployment to all 120 properties by Q3. The CFO told me afterward that the statistical evidence table was what won him over — he said he'd seen too many pitches based on gut feeling and cherry-picked numbers."*

Elena nods. *"The brand promise discussion was important too. Now we know where we stand — and we have a path to actually earning that 9.0 rating instead of just advertising it."*

Marcus is already typing on his phone. *"I'm setting up the monitoring dashboard. We'll track all three metrics across every property as we roll out. Quarterly board updates, same rigor."*

Priya looks at both of them. *"This is what data-driven decision-making looks like. Not just having the data — but knowing the right questions to ask and the right tests to answer them."*

She picks up her bag and heads for the elevator. There's already another decision waiting on her desk — but this time, she knows exactly how to approach it.

---

## Interaction Log

### AI Tool Usage
*Did you use any AI tools (ChatGPT, GitHub Copilot, Claude, etc.)? If so, describe what you used them for and how they helped.*

[Used Gemini's google collab assistant for code deployment and assistance with logic references]

### Statistical Tests Used
*List each statistical test you performed, which part it appeared in, and explain why that test was appropriate for the question being asked.*

[One-sample T, Independent T-test, Paired T-tests]

### Interpretation Approach
*How did you translate statistical results into business recommendations? What was your process for going from p-values to actionable advice?*

[Besides utilizing the P-values given from calculations to reject or fail to reject the null hypothesis, I made sure to include business specific considerations that go beyond surface level results. Additionally, I interpreted p-values in laymen terms for stakeholders and explained how data suggest the mobile format was superior in different areas]

---

## Submission Checklist

- [x] All code cells run without errors
- [x] All five parts completed
- [x] Hypotheses clearly stated for each test
- [x] Correct test chosen for each scenario (with justification)
- [x] P-values and decisions reported at α = 0.05
- [x] Mastery questions answered thoughtfully
- [x] Business interpretations provided (not just statistics)
- [x] Board recommendation complete and justified
- [x] Interaction log completed
- [x] Video walkthrough recorded (3-5 minutes)

---
*ISM4641 Python for Business Analytics — Week 11 Integrative Assignment*